In [ ]:
from datetime import datetime, timedelta
import json
from pathlib import Path

from matplotlib.dates import DateFormatter
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy.stats import boxcox
from scipy.optimize import minimize
from statsmodels.tsa.arima.model import ARIMA

## Dynamic Regression Baseline Model

In [ ]:
class BoxCoxScaler:
    def __init__(self):
        self._lambda: float | None = None

    @property
    def is_fit(self):
        return self._lambda is not None
    
    def fit_transform(self, y: pl.Series) -> pl.Series:
        y_t, _lambda = boxcox(y.to_numpy())
        self._lambda = _lambda
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    
    def transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit
        y_t = boxcox(y.to_numpy(), lmbda=self._lambda)
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    
    def inverse_transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit

        if self._lambda == 0:
            y_t = np.exp(y.to_numpy())
        else:
            y_t = (y.to_numpy() * self._lambda + 1) ** (1 / self._lambda)
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)


class FourierRegressionModel:
    def __init__(self, include_bias: bool = True):
        self.include_bias = include_bias

    def fit(self, X: np.ndarray, y: np.ndarray):
        if self.include_bias:
            X = np.hstack([np.ones((X.shape[0], 1)), X])

        def f_(theta: np.ndarray):
            y_hat = np.dot(X, theta)
            return np.mean((y - y_hat) ** 2)    

        x0 = np.array([0.5 for _ in range(X.shape[1])])
        result = minimize(f_, x0=x0)
        if not result.success:
            raise ValueError(result.message)
        self.theta_ = result.x

        return self

    def predict(self, X: np.ndarray):
        if self.include_bias:
            X = np.hstack([np.ones((X.shape[0], 1)), X])
        return np.dot(X, self.theta_)


class DynamicRegressionModel:
    def __init__(self):
        ...

    def fit(self):
        ...


    def predict(self,):
        ...



## PJM Dataset

In [ ]:
PJM_SITE_NAME = "PJMW"
PJM_DATA_FREQUENCY = "1h"

INPUT_PATH = Path("../../data/pjm")
OUTPUT_PATH = Path(f"../../results/pjm/naive/{PJM_SITE_NAME}")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

data_file_name = f"{PJM_SITE_NAME}_hourly_processed.pq"
data_file_path = INPUT_PATH / data_file_name
SITE_DF = pl.read_parquet(data_file_path).sort(by="timestamp")

### Configure

In [ ]:
MAX_FOURIER_HARMONIC = 5
SEASONAL_PERIOD = int(24 * 7)
TARGET_COL = f"{PJM_SITE_NAME}_MW"

In [ ]:
def compute_fourier_features(df: pl.DataFrame) -> pl.DataFrame:
    cycles = [
        ("hour", pl.col("timestamp").dt.hour(), 24),
        ("day", pl.col("timestamp").dt.day(), 7),
        ("month", pl.col("timestamp").dt.month(), 12),
    ]
    col_expr = {}
    for unit, t_expr, period in cycles:
        for k in range(1, MAX_FOURIER_HARMONIC + 1):
            angle = (2 * np.pi * k * t_expr) / period
            col_expr[f"sin_{k}_{unit}"] = np.sin(angle)
            col_expr[f"cos_{k}_{unit}"] = np.cos(angle)
    return df.with_columns(**col_expr)

In [ ]:
train_df = compute_fourier_features(SITE_DF)
train_df.head()

In [ ]:
scaler = BoxCoxScaler()

X = train_df[
    [f"sin_{k}_hour" for k in range(1, MAX_FOURIER_HARMONIC + 1)]
    + [f"cos_{k}_hour" for k in range(1, MAX_FOURIER_HARMONIC + 1)]
    + [f"sin_{k}_day" for k in range(1, MAX_FOURIER_HARMONIC + 1)]
    + [f"cos_{k}_day" for k in range(1, MAX_FOURIER_HARMONIC + 1)]
]
y = train_df[TARGET_COL]
y_scaled = scaler.fit_transform(y)

model = FourierRegressionModel(include_bias=True)
model = model.fit(X.to_numpy(), y_scaled.to_numpy())

In [ ]:
y_hat = model.predict(X.to_numpy())

In [ ]:
residuals = (y_scaled - y_hat)
residuals_diff = residuals[1:] - residuals[:-1]

plt.plot(
    np.arange(len(residuals_diff)),
    residuals_diff,
    lw=0.5,
    alpha=0.5
)
plt.axhline(0, color="black", ls="--", lw=0.5, alpha=0.5)

In [ ]:
model = ARIMA(endog=y[-2000:], exog=X[-2000:], order=(1, 1, 1))
results = model.fit()

In [ ]:
fitted_values = results.predict()

In [ ]:
plt.plot(y[-2000:])
plt.plot(fitted_values)